# DMOSpeech 2 — EXP-032: how much variety each teacher pass buys (against the paper's t_switch = 0.25)

- **Task ID:** `EVAL-025` (development study, preregistered before any synthesis). No handover method is tested: the results set the target for the next handover design.
- **Why:** DMOSpeech 2 runs its teacher early to restore the prosodic variety its fast student loses (paper, Sections 3.3 and 4.3.3). The teacher's passes are the price of that variety.
- **The 100% reference (`P25`):** the paper's evaluated setting, t_switch = 0.25, as its Algorithm 2 describes: 14 teacher steps on the 32-interval grid, then 2 student steps (30 passes).
- **Measured on the same cases:**
  - `STU` (student alone, 4 passes);
  - `W1`, `W2`, `W3`, `W4`, `W8` (teacher to Step 1-8 on the released 16-interval grid, then the student; 7-21 passes; `W8` is the thesis's version of 0.25);
  - `P25R` (the paper's 14 teacher steps with the released code's handover, 32 passes);
  - `TCH` (teacher alone, 64 passes; quality stage only).
- **Prompts:** clean and `noisy15` (white noise at 15 dB).
- **Cases:** development speakers 1-30 × 2 conditions for quality (60 cases), and speakers 1-20 × 12 seeds × 2 conditions for prosodic variety.
- **Tensor bank:** the student's draft, the teacher's estimates and four outputs (generated frames, float16) for speakers 1-5, about 120 MB, saved to Drive in a separate zip for offline screening of new handover designs.
- **Estimated T4 runtime:** about 3 hours with `STAGE = "all"` (the default). It resumes from Drive if the runtime disconnects: just **Run all** again.
- **When it finishes,** the results zip is saved to Drive (`MyDrive/exp032_variety_per_pass/`) and also downloaded; the tensor-bank zip is saved to the same Drive folder.


In [ ]:
import os
import subprocess

# GPU checked with nvidia-smi, not PyTorch: the notebook kernel then holds no PyTorch or CUDA state in RAM,
# so the runner (two TTS models, Whisper large-v3, WavLM and UTMOS) has all of it. Results are unaffected.
q = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
assert q.returncode == 0 and q.stdout.strip(), "No GPU found. Switch the runtime to T4 GPU (Runtime > Change runtime type)."
gpu_name = q.stdout.strip().splitlines()[0]
print("GPU:", gpu_name)
assert "T4" in gpu_name, f"Expected a Tesla T4 (found {gpu_name}): the smoke test reproduces EXP-027's T4 scores."

from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-032-variety-per-pass/results"
DRIVE_DIR = "/content/drive/MyDrive/exp032_variety_per_pass"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(DRIVE_CACHE, exist_ok=True)
print("Drive directories ready:", DRIVE_DIR)

## 1. Clone Repositories, Install Dependencies & Verify Preregistration

Clones `Shonget-DMOSpeech2` (pinned commit `8a7868fc...`) and `Final-Thesis-Setup` (branch `rhnl-3`), which holds `code/colab/run_variety_per_pass.py` and `research/experiments/EXP-032-variety-per-pass/preregistration.json`.


In [ ]:
import json
import os
import subprocess
import sys

THESIS_BRANCH = "rhnl-3"
DMO_COMMIT = "8a7868fc3345702a586173447f4869205b501f26"
DMO_REPO = "https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git"
THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
STAGE = "all"  # runs quality (60 cases) then diversity (20 x 12 x 2 conditions); resumes from Drive. Other options: "quality" or "diversity"

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
thesis_url = f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git"

if not os.path.exists(DMO_ROOT):
    subprocess.run(["git", "clone", DMO_REPO, DMO_ROOT], check=True)
subprocess.run(["git", "-C", DMO_ROOT, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", DMO_ROOT, "checkout", "-q", DMO_COMMIT], check=True)

if not os.path.exists(THESIS_ROOT):
    subprocess.run(["git", "clone", "-b", THESIS_BRANCH, thesis_url, THESIS_ROOT], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "fetch", "-q", "origin", THESIS_BRANCH], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "checkout", "-q", THESIS_BRANCH], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "pull", "-q", "origin", THESIS_BRANCH], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy==2.2.3", "pandas==2.2.3"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{DMO_ROOT}/requirements.txt"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torchdiffeq", "vocos", "speechbrain", "transformers", "jiwer",
    "soundfile", "librosa", "pyarrow", "scipy", "jieba", "accelerate", "huggingface_hub"
], check=True)

prereg_path = f"{THESIS_ROOT}/research/experiments/EXP-032-variety-per-pass/preregistration.json"
assert os.path.exists(prereg_path), f"Missing {prereg_path} on branch {THESIS_BRANCH}"
prereg = json.loads(open(prereg_path, encoding="utf-8").read())
print(f"Preregistration verified: sha256={prereg['sha256'][:12]}..., qual={prereg['n_qual_cases']}, div={prereg['n_div_speakers']}x{prereg['div_seeds']}")

## 2. Checkpoints & Seed-TTS English Audio (Drive Cache Supported)


In [ ]:
import hashlib
import os
import shutil
import subprocess

DMO_ROOT = "/content/Shonget-DMOSpeech2"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
os.makedirs(f"{DMO_ROOT}/ckpts", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/ckpts", exist_ok=True)

CKPTS = {
    "model_85000.pt": (
        "https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt",
        "e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c",
    ),
    "model_1500.pt": (
        "https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt",
        "6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2",
    ),
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

for name, (url, expected) in CKPTS.items():
    local_p = f"{DMO_ROOT}/ckpts/{name}"
    cache_p = f"{DRIVE_CACHE}/ckpts/{name}"
    if not os.path.exists(local_p) and os.path.exists(cache_p):
        shutil.copy2(cache_p, local_p)
    if not os.path.exists(local_p):
        subprocess.run(["wget", "-c", url, "-O", local_p], check=True)
    digest = sha256_file(local_p)
    assert digest == expected, f"{name} SHA-256 mismatch: got {digest}"
    if not os.path.exists(cache_p):
        shutil.copy2(local_p, cache_p)
    print(f"{name}: SHA-256 verified")

data_dir = f"{DMO_ROOT}/data/seed-tts-eval"
probe_wav = f"{data_dir}/en/prompt-wavs/common_voice_en_10933823.wav"
cache_data = f"{DRIVE_CACHE}/seed-tts-eval"
if (not os.path.exists(probe_wav) or os.path.getsize(probe_wav) < 1000) and os.path.exists(f"{cache_data}/en/meta.lst"):
    shutil.copytree(cache_data, data_dir, dirs_exist_ok=True)
if not os.path.exists(probe_wav) or os.path.getsize(probe_wav) < 1000:
    from huggingface_hub import snapshot_download
    shutil.rmtree(data_dir, ignore_errors=True)
    snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir=data_dir)
assert os.path.exists(probe_wav) and os.path.getsize(probe_wav) >= 1000, "Seed-TTS audio missing"
print("Seed-TTS audio verified [OK]")

## 3. Restore Any Partial Results from Google Drive


In [ ]:
import csv
import os
import shutil

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-032-variety-per-pass/results"
DRIVE_DIR = "/content/drive/MyDrive/exp032_variety_per_pass"
os.makedirs(OUT_DIR, exist_ok=True)

for rel in ("quality.csv", "diversity.csv", "smoke/quality.csv", "smoke/diversity.csv", "smoke/smoke_check.json"):
    src = os.path.join(DRIVE_DIR, rel)
    dst = os.path.join(OUT_DIR, rel)
    if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst)):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Restored from Drive:", rel)

restored = 0
bank_src = os.path.join(DRIVE_DIR, "bank")
for root, _, fns in os.walk(bank_src):
    for fn in fns:
        src = os.path.join(root, fn)
        dst = os.path.join(OUT_DIR, "bank", os.path.relpath(src, bank_src))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            restored += 1
if restored:
    print(f"Restored {restored} tensor-bank files from Drive")

for fname, target in (("quality.csv", 60), ("diversity.csv", 40)):
    p = os.path.join(OUT_DIR, fname)
    n = sum(1 for _ in csv.DictReader(open(p, encoding="utf-8"))) if os.path.exists(p) else 0
    print(f"{fname}: {n} / {target} rows")


## 4. Pre-Flight Smoke Test

Verifies the preregistration hashes and exact identities: the split student loop and the paper setting's student repeated (both `<= 1e-5`), and the 32-interval teacher function against the 16-interval trajectory on the same grid (`<= 1e-4`). It reproduces EXP-027's `W4` scores (`<= 0.01`), then runs development speaker 1 under both conditions and checks `STU`, `W4` and `W8` against EXP-031's stored scores (`<= 0.01`); the paper's settings are compared with EXP-017's stored scores and recorded. It ends with a two-seed variety run on one speaker that writes and reloads one tensor-bank file.

If it stops with exit code -9 (`died with <Signals.SIGKILL: 9>`), the machine ran out of RAM: use **Runtime > Disconnect and delete runtime**, then **Run all**.


In [ ]:
import json
import os
import subprocess
import sys

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-032-variety-per-pass/results"
RUNNER = f"{THESIS_ROOT}/code/colab/run_variety_per_pass.py"

def ram_gb():
    info = dict(l.split(":", 1) for l in open("/proc/meminfo") if ":" in l)
    return int(info["MemTotal"].split()[0]) / 2**20, int(info["MemAvailable"].split()[0]) / 2**20

total, avail = ram_gb()
print(f"RAM: {avail:.1f} GB available of {total:.1f} GB")
if avail < 10:
    print("WARNING: little RAM is free. If the job is killed (exit code -9), use Runtime > Disconnect and delete runtime, then Run all.")
LEAN = {"MALLOC_ARENA_MAX": "2", "PYTHONUNBUFFERED": "1"}  # fewer allocator arenas while the large models load; results are unaffected

os.chdir(THESIS_ROOT)
proc = subprocess.Popen([sys.executable, "-u", RUNNER, "smoke", "--dmo_root", DMO_ROOT, "--out_dir", OUT_DIR],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, **LEAN})
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
if code in (-9, 137):
    raise SystemExit("The smoke test was killed (exit code -9): the machine ran out of RAM. "
                     "Use Runtime > Disconnect and delete runtime, then Run all again.")
assert code == 0, f"Smoke test failed with code {code}"
print(json.dumps(json.loads(open(f"{OUT_DIR}/smoke/smoke_check.json", encoding="utf-8").read()), indent=2))

## 5. Run Synthesis (`all` = about 3 hours: quality ~45 min, then variety ~2 h 20 min)


In [ ]:
import os
import shutil
import subprocess
import sys
import threading

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-032-variety-per-pass/results"
DRIVE_DIR = "/content/drive/MyDrive/exp032_variety_per_pass"
RUNNER = f"{THESIS_ROOT}/code/colab/run_variety_per_pass.py"
STAGE = str(globals().get("STAGE", "all")).strip().lower()  # set STAGE in the setup cell (Step 1), not here
assert STAGE in ("quality", "diversity", "all"), f"STAGE must be 'quality', 'diversity' or 'all', not {STAGE!r}"
print("Stage:", STAGE, {"all": "(about 3 hours)", "quality": "(about 45 min)", "diversity": "(about 2 h 20 min)"}[STAGE])

def sync_to_drive():
    if not os.path.exists(OUT_DIR):
        return
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            src = os.path.join(root, fn)
            if not os.path.exists(src) or fn.startswith("tmp_"):
                continue
            rel = os.path.relpath(src, OUT_DIR)
            dst = os.path.join(DRIVE_DIR, rel)
            if (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)
                    and os.path.getmtime(dst) >= os.path.getmtime(src)):
                continue  # unchanged since the last sync (tensor-bank files are written once)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)

stop_evt = threading.Event()
def sync_loop():
    while not stop_evt.wait(60):
        try:
            sync_to_drive()
        except Exception as exc:
            print("Drive sync warning:", exc)

t = threading.Thread(target=sync_loop, daemon=True)
t.start()
def ram_gb():
    info = dict(l.split(":", 1) for l in open("/proc/meminfo") if ":" in l)
    return int(info["MemTotal"].split()[0]) / 2**20, int(info["MemAvailable"].split()[0]) / 2**20

total, avail = ram_gb()
print(f"RAM: {avail:.1f} GB available of {total:.1f} GB")
if avail < 10:
    print("WARNING: little RAM is free. If the job is killed (exit code -9), use Runtime > Disconnect and delete runtime, then Run all.")
LEAN = {"MALLOC_ARENA_MAX": "2", "PYTHONUNBUFFERED": "1"}  # fewer allocator arenas while the large models load; results are unaffected

os.chdir(THESIS_ROOT)
code = None
try:
    proc = subprocess.Popen([sys.executable, "-u", RUNNER, "run", "--dmo_root", DMO_ROOT, "--out_dir", OUT_DIR, "--stage", STAGE],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, **LEAN})
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
finally:
    stop_evt.set()
    t.join(timeout=5)
    sync_to_drive()
if code in (-9, 137):
    raise SystemExit("The run was killed (exit code -9): the machine ran out of RAM. Use Runtime > Disconnect and delete runtime, "
                     "then Run all: Step 3 restores the finished cases from Drive and the run resumes from them.")
assert code == 0, f"Run failed with code {code}"
print("Run finished and synced to Drive.")

## 6. Analyze & Export Results ZIP (and the Tensor-Bank ZIP)

Runs the full analysis once `quality.csv` (60 rows) and `diversity.csv` (40 rows) are complete, otherwise a partial one. The results zip (CSVs, report, manifest and the bank's hash index; no tensors) is saved to Drive first, then downloaded; if the browser tab has lost its connection, take it from Drive. The tensor bank is zipped separately to Drive only (`DMOSpeech2_EXP032_Tensor_Bank.zip`, about 120 MB); download it from Drive when it is asked for.


In [ ]:
import csv
import os
import shutil
import subprocess
import sys
import zipfile
from google.colab import files

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-032-variety-per-pass/results"
DRIVE_DIR = "/content/drive/MyDrive/exp032_variety_per_pass"
RUNNER = f"{THESIS_ROOT}/code/colab/run_variety_per_pass.py"
ZIP_NAME = "DMOSpeech2_EXP032_Variety_Per_Pass_Results.zip"
ZIP_PATH = f"/content/{ZIP_NAME}"
BANK_ZIP_NAME = "DMOSpeech2_EXP032_Tensor_Bank.zip"
BANK_ZIP_PATH = f"/content/{BANK_ZIP_NAME}"

q_rows = sum(1 for _ in csv.DictReader(open(f"{OUT_DIR}/quality.csv", encoding="utf-8"))) if os.path.exists(f"{OUT_DIR}/quality.csv") else 0
d_rows = sum(1 for _ in csv.DictReader(open(f"{OUT_DIR}/diversity.csv", encoding="utf-8"))) if os.path.exists(f"{OUT_DIR}/diversity.csv") else 0
cmd = [sys.executable, RUNNER, "analyze", "--out_dir", OUT_DIR]
if q_rows < 60 or d_rows < 40:
    cmd.append("--allow_partial")
os.chdir(THESIS_ROOT)
subprocess.run(cmd, check=True)
print(open(f"{OUT_DIR}/VARIETY_PER_PASS_REPORT.md", encoding="utf-8").read())

for root, _, fns in os.walk(OUT_DIR):
    for fn in fns:
        src = os.path.join(root, fn)
        if fn.startswith("tmp_"):
            continue
        rel = os.path.relpath(src, OUT_DIR)
        dst = os.path.join(DRIVE_DIR, rel)
        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src) and os.path.getmtime(dst) >= os.path.getmtime(src):
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

def is_bank(rel):
    return "bank" in rel.replace(os.sep, "/").split("/")[:-1]  # bank/... and smoke/bank/...: tensors stay out of this zip

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fns in os.walk(OUT_DIR):
        for fn in fns:
            src = os.path.join(root, fn)
            rel = os.path.relpath(src, OUT_DIR)
            if fn.startswith("tmp_") or is_bank(rel):
                continue
            zf.write(src, arcname=rel)
shutil.copy2(ZIP_PATH, os.path.join(DRIVE_DIR, ZIP_NAME))
print("Packaged:", ZIP_PATH)
print("Saved to Drive:", os.path.join(DRIVE_DIR, ZIP_NAME))

bank_dir = os.path.join(OUT_DIR, "bank")
if os.path.isdir(bank_dir):
    with zipfile.ZipFile(BANK_ZIP_PATH, "w", zipfile.ZIP_STORED) as zf:  # .npz files are already compressed
        for root, _, fns in os.walk(bank_dir):
            for fn in fns:
                src = os.path.join(root, fn)
                zf.write(src, arcname=os.path.relpath(src, OUT_DIR))
    shutil.copy2(BANK_ZIP_PATH, os.path.join(DRIVE_DIR, BANK_ZIP_NAME))
    print(f"Tensor bank saved to Drive: {os.path.join(DRIVE_DIR, BANK_ZIP_NAME)} ({os.path.getsize(BANK_ZIP_PATH) / 2**20:.1f} MiB)")

files.download(ZIP_PATH)
